# 01 — Object Detection with YOLOv8: Data + Training

One notebook that takes Pascal VOC from download to a trained YOLOv8 detector:

1. Download VOC (2007 + 2012)
2. Convert XML annotations to YOLO format and split the data
3. Explore class/box statistics and verify boxes visually
4. Run k-means anchor analysis (educational — YOLOv8 is anchor-free)
5. Write the Ultralytics dataset YAML
6. Train YOLOv8n and validate the best checkpoint
7. Plot loss and mAP curves

**Run on Google Colab:** upload this notebook, then
`Runtime → Change runtime type → GPU`, then `Runtime → Run all`.

**Quick run (default):** only VOC2007 is downloaded (~884 MB), 800 train /
200 val images are used, and training runs for 10 epochs. Set
`QUICK_RUN = False` in the setup cell for the full official split
(~2.8 GB, 16,551 train / 4,952 val images) and 50 epochs.

In [ ]:
%pip install -q ultralytics scikit-learn pandas pyyaml

## 1. Setup

In [ ]:
import random
import shutil
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageDraw

DATA_DIR = Path("data").resolve()
MODELS_DIR = Path("models").resolve()
RESULTS_DIR = Path("results").resolve()
for directory in (DATA_DIR, MODELS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# ---------------- toggles ----------------
QUICK_RUN = True
RUN_TRAINING = True

MAX_TRAIN = 800 if QUICK_RUN else None
MAX_VAL = 200 if QUICK_RUN else None

MODEL_NAME = "yolov8n.pt"
RUN_NAME = "yolov8n_voc"
IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 10 if QUICK_RUN else 50
SEED = 42

VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat",
    "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

BASE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
VOC_URLS = {
    "VOCtrainval_06-Nov-2007.zip": BASE_URL + "VOCtrainval_06-Nov-2007.zip",
    "VOCtest_06-Nov-2007.zip": BASE_URL + "VOCtest_06-Nov-2007.zip",
    "VOCtrainval_11-May-2012.zip": BASE_URL + "VOCtrainval_11-May-2012.zip",
}
VOC2007_URLS = {name: url for name, url in VOC_URLS.items() if "2007" in name}

OFFICIAL_SPLITS = {
    "train": [("2012", "train"), ("2012", "val"), ("2007", "train"), ("2007", "val")],
    "val": [("2007", "test")],
}
QUICK_SPLITS = {
    "train": [("2007", "train"), ("2007", "val")],
    "val": [("2007", "test")],
}

COLORS = [
    (230, 25, 75), (60, 180, 75), (255, 225, 25), (0, 130, 200),
    (245, 130, 48), (145, 30, 180), (70, 240, 240), (240, 50, 230),
    (210, 245, 60), (250, 190, 212), (0, 128, 128), (220, 190, 255),
    (170, 110, 40), (255, 250, 200), (128, 0, 0), (170, 255, 195),
    (128, 128, 0), (255, 215, 180), (0, 0, 128), (128, 128, 128),
]

print("Quick run:", QUICK_RUN, "| epochs:", EPOCHS)
print("Data directory:", DATA_DIR)

## 2. Download Pascal VOC

Archives come from the Ultralytics asset mirror. Already-downloaded files are
skipped, so this cell is safe to re-run.

In [ ]:
def download_file(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already downloaded: {destination.name}")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {destination.name} ...")
    urllib.request.urlretrieve(url, destination)
    return destination


def download_voc(data_dir, include_2012=True):
    data_dir = Path(data_dir)
    urls = VOC_URLS if include_2012 else VOC2007_URLS
    for name, url in urls.items():
        archive = download_file(url, data_dir / "downloads" / name)
        marker = data_dir / "VOCdevkit" / f".{name}.extracted"
        if not marker.exists():
            print(f"Extracting {name} ...")
            with zipfile.ZipFile(archive) as archive_file:
                archive_file.extractall(data_dir)
            marker.parent.mkdir(parents=True, exist_ok=True)
            marker.touch()
    return data_dir / "VOCdevkit"


devkit = download_voc(DATA_DIR, include_2012=not QUICK_RUN)
print("VOCdevkit:", devkit)

## 3. Convert XML annotations to YOLO format

VOC stores boxes as `xmin, ymin, xmax, ymax` in pixels. YOLO needs
`class x_center y_center width height`, normalized to `[0, 1]`.
Images are prefixed with the VOC year to avoid filename collisions.

In [ ]:
def convert_box(size, box):
    width, height = size
    x_min, x_max, y_min, y_max = box
    return (
        ((x_min + x_max) / 2.0 - 1.0) / width,
        ((y_min + y_max) / 2.0 - 1.0) / height,
        (x_max - x_min) / width,
        (y_max - y_min) / height,
    )


def convert_annotation(xml_path, label_path, class_names=VOC_CLASSES):
    root = ET.parse(xml_path).getroot()
    size_node = root.find("size")
    image_size = (
        int(size_node.find("width").text),
        int(size_node.find("height").text),
    )

    lines = []
    for obj in root.iter("object"):
        class_name = obj.find("name").text
        if class_name not in class_names:
            continue
        difficult = obj.find("difficult")
        if difficult is not None and int(difficult.text) == 1:
            continue

        box_node = obj.find("bndbox")
        box = [
            float(box_node.find(key).text)
            for key in ("xmin", "xmax", "ymin", "ymax")
        ]
        coords = [min(max(value, 0.0), 1.0) for value in convert_box(image_size, box)]
        if coords[2] <= 0.0 or coords[3] <= 0.0:
            continue

        class_id = class_names.index(class_name)
        values = [str(class_id)] + [f"{value:.6f}" for value in coords]
        lines.append(" ".join(values))

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text(
        "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
    )
    return len(lines)


def convert_subset(voc_root, data_dir, year, image_set, split, max_images=None):
    year_dir = Path(voc_root) / f"VOC{year}"
    ids_path = year_dir / "ImageSets" / "Main" / f"{image_set}.txt"
    image_ids = ids_path.read_text(encoding="utf-8").strip().split()
    if max_images is not None:
        image_ids = image_ids[:max_images]

    images_out = Path(data_dir) / "images" / split
    labels_out = Path(data_dir) / "labels" / split
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    count = 0
    for image_id in image_ids:
        source_image = year_dir / "JPEGImages" / f"{image_id}.jpg"
        source_xml = year_dir / "Annotations" / f"{image_id}.xml"
        if not source_image.exists():
            continue

        stem = f"{year}_{image_id}"
        shutil.copy2(source_image, images_out / f"{stem}.jpg")

        destination_label = labels_out / f"{stem}.txt"
        if source_xml.exists():
            convert_annotation(source_xml, destination_label)
        else:
            destination_label.write_text("", encoding="utf-8")
        count += 1
    return count


def reset_split_dirs(data_dir, splits):
    for split in splits:
        for folder in ("images", "labels"):
            directory = Path(data_dir) / folder / split
            if directory.exists():
                shutil.rmtree(directory)
            directory.mkdir(parents=True, exist_ok=True)


splits = OFFICIAL_SPLITS if not QUICK_RUN else QUICK_SPLITS
reset_split_dirs(DATA_DIR, list(splits))

summary = {}
for split, subsets in splits.items():
    remaining = MAX_TRAIN if split == "train" else MAX_VAL
    total = 0
    for year, image_set in subsets:
        converted = convert_subset(
            devkit, DATA_DIR, year, image_set, split, max_images=remaining
        )
        total += converted
        if remaining is not None:
            remaining -= converted
            if remaining <= 0:
                break
    summary[split] = total

print("Converted images:", summary)

## 4. Save reproducible split lists

In [ ]:
splits_dir = DATA_DIR / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

for split in ("train", "val"):
    images = sorted((DATA_DIR / "images" / split).glob("*.jpg"))
    out_path = splits_dir / f"{split}.txt"
    out_path.write_text(
        "\n".join(path.relative_to(DATA_DIR).as_posix() for path in images) + "\n",
        encoding="utf-8",
    )
    print(f"{split}: {len(images)} images -> {out_path}")

## 5. Dataset statistics

Each label file has one line per object. Empty label files are valid and
represent images with no annotated objects (background/negative samples).

In [ ]:
def load_label_file(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros((0, 5), dtype=np.float32)
    return np.loadtxt(path, ndmin=2, dtype=np.float32)


def dataset_stats(data_dir, split):
    label_files = sorted((Path(data_dir) / "labels" / split).glob("*.txt"))
    class_counts = Counter()
    boxes_per_image = []
    box_sizes = []

    for label_file in label_files:
        labels = load_label_file(label_file)
        boxes_per_image.append(len(labels))
        for class_id, _, _, width, height in labels:
            class_counts[VOC_CLASSES[int(class_id)]] += 1
            box_sizes.append((float(width), float(height)))

    return {
        "images": len(label_files),
        "boxes": int(sum(boxes_per_image)),
        "images_with_objects": int(sum(1 for count in boxes_per_image if count > 0)),
        "mean_boxes": float(np.mean(boxes_per_image)) if boxes_per_image else 0.0,
        "max_boxes": int(max(boxes_per_image, default=0)),
        "class_counts": class_counts,
        "box_sizes": np.array(box_sizes) if box_sizes else np.zeros((0, 2)),
    }


train_stats = dataset_stats(DATA_DIR, "train")
val_stats = dataset_stats(DATA_DIR, "val")

print(
    f"Train: {train_stats['images']} images | {train_stats['boxes']} boxes | "
    f"{train_stats['mean_boxes']:.2f} boxes/image | max {train_stats['max_boxes']}"
)
print(
    f"Val:   {val_stats['images']} images | {val_stats['boxes']} boxes | "
    f"{val_stats['mean_boxes']:.2f} boxes/image"
)
print("\nTop classes (train):")
for name, count in train_stats["class_counts"].most_common(10):
    print(f"  {name:14s} {count}")

In [ ]:
fig, (ax_counts, ax_sizes) = plt.subplots(1, 2, figsize=(14, 5))

counts = [train_stats["class_counts"].get(name, 0) for name in VOC_CLASSES]
ax_counts.barh(VOC_CLASSES, counts, color="#3b82f6")
ax_counts.invert_yaxis()
ax_counts.set_xlabel("Instances")
ax_counts.set_title(f"Train class distribution ({train_stats['boxes']} boxes)")
ax_counts.grid(axis="x", alpha=0.3)

sizes = train_stats["box_sizes"]
if len(sizes) > 0:
    ax_sizes.scatter(sizes[:, 0], sizes[:, 1], s=8, alpha=0.25, color="#ef4444")
ax_sizes.set_xlabel("Normalized width")
ax_sizes.set_ylabel("Normalized height")
ax_sizes.set_title("Box size distribution")
ax_sizes.set_xlim(0, 1)
ax_sizes.set_ylim(0, 1)
ax_sizes.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Visual check: do boxes align with objects?

If the rectangles are misplaced, the coordinate conversion is wrong. This is the
key sanity check before training.

In [ ]:
def draw_boxes(image_path, label_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)

    for (
        class_id,
        x_center,
        y_center,
        box_width,
        box_height,
    ) in load_label_file(label_path):
        class_id = int(class_id)
        x1 = (x_center - box_width / 2.0) * width
        y1 = (y_center - box_height / 2.0) * height
        x2 = (x_center + box_width / 2.0) * width
        y2 = (y_center + box_height / 2.0) * height

        color = COLORS[class_id % len(COLORS)]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

        label = VOC_CLASSES[class_id]
        text_box = draw.textbbox((x1, y1), label)
        draw.rectangle(
            [text_box[0], text_box[1] - 2, text_box[2] + 2, text_box[3] + 2],
            fill=color,
        )
        draw.text((x1, y1 - 12), label, fill="white")

    return image


images = sorted((DATA_DIR / "images" / "train").glob("*.jpg"))
sample = random.Random(42).sample(images, min(6, len(images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image_path in zip(axes.ravel(), sample):
    label_path = DATA_DIR / "labels" / "train" / f"{image_path.stem}.txt"
    ax.imshow(draw_boxes(image_path, label_path))
    ax.set_title(image_path.name)
    ax.axis("off")

for ax in axes.ravel()[len(sample):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Inspect one raw YOLO label file

In [ ]:
example_image = sample[0]
example_label = DATA_DIR / "labels" / "train" / f"{example_image.stem}.txt"
print(example_image)
print(example_label)
print(example_label.read_text(encoding="utf-8") or "(empty label file)")

## 7. Anchor analysis (educational)

Classic YOLO detectors use pre-defined anchor boxes. We cluster the training
boxes with k-means to find anchors that fit this dataset, and plot the elbow
curve plus the mean best IoU.

> **Note:** YOLOv8 is **anchor-free** and does not use these anchors for
> training. This section teaches the classic anchor-design workflow from the
> project notes; it is not passed to the YOLOv8 trainer.

In [ ]:
def kmeans_anchors(sizes, k, iterations=30, seed=SEED):
    rng = np.random.default_rng(seed)
    centers = sizes[rng.choice(len(sizes), k, replace=False)].astype(np.float64)

    for _ in range(iterations):
        distances = ((sizes[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        assignments = distances.argmin(axis=1)
        new_centers = np.array([
            sizes[assignments == index].mean(axis=0)
            if np.any(assignments == index) else centers[index]
            for index in range(k)
        ])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers

    return centers


def mean_best_iou(sizes, anchors):
    intersection = (
        np.minimum(sizes[:, None, 0], anchors[None, :, 0])
        * np.minimum(sizes[:, None, 1], anchors[None, :, 1])
    )
    box_area = sizes[:, None, 0] * sizes[:, None, 1]
    anchor_area = anchors[None, :, 0] * anchors[None, :, 1]
    union = box_area + anchor_area - intersection
    return float((intersection / union).max(axis=1).mean())


train_sizes = train_stats["box_sizes"]
assert len(train_sizes) > 0, "No boxes found; run the data cells first."

max_k = min(9, len(train_sizes))
k_values = list(range(1, max_k + 1))
inertias = []
ious = []
anchor_models = {}

for k in k_values:
    centers = kmeans_anchors(train_sizes, k)
    distances = ((train_sizes[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    inertias.append(float(distances.min(axis=1).sum()))
    ious.append(mean_best_iou(train_sizes, centers))
    anchor_models[k] = centers

best_k = max_k
anchors_px = anchor_models[best_k] * IMG_SIZE
anchors_px = anchors_px[np.argsort(anchors_px[:, 0] * anchors_px[:, 1])]

print(f"Anchors for k={best_k} (pixels at {IMG_SIZE}px):")
for index, (width, height) in enumerate(anchors_px, start=1):
    print(f"  {index}: {width:6.1f} x {height:6.1f}")

fig, (ax_elbow, ax_iou) = plt.subplots(1, 2, figsize=(12, 4))
ax_elbow.plot(k_values, inertias, marker="o", color="#3b82f6")
ax_elbow.set_xlabel("k (number of anchors)")
ax_elbow.set_ylabel("Inertia")
ax_elbow.set_title("Elbow curve")
ax_elbow.grid(alpha=0.3)

ax_iou.plot(k_values, ious, marker="o", color="#10b981")
ax_iou.set_xlabel("k (number of anchors)")
ax_iou.set_ylabel("Mean best IoU")
ax_iou.set_title("Anchor quality")
ax_iou.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Write the Ultralytics dataset YAML

Ultralytics needs an absolute dataset `path` plus train/val image folders.
`test` aliases `val` because VOC2007 test is the only annotated holdout.

In [ ]:
DATASET_YAML = DATA_DIR / "dataset.yaml"
dataset_config = {
    "path": str(DATA_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/val",
    "names": {index: name for index, name in enumerate(VOC_CLASSES)},
}
DATASET_YAML.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8"
)
print(DATASET_YAML.read_text(encoding="utf-8"))

## 9. Train YOLOv8

- Baseline model: `yolov8n.pt` (nano, fastest)
- Quick run: 10 epochs on ~800 images
- Full run: set `QUICK_RUN = False` above, then 50 epochs on the official split

Training artifacts land in `results/runs/<RUN_NAME>/`; the best checkpoint is
copied to `models/<RUN_NAME>_best.pt`.

In [ ]:
if RUN_TRAINING:
    import torch
    from ultralytics import YOLO

    device = 0 if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    model = YOLO(MODEL_NAME)
    model.train(
        data=str(DATASET_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=device,
        workers=2,
        patience=15,
        seed=SEED,
        project=str(RESULTS_DIR / "runs"),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
    )

    RUN_DIR = RESULTS_DIR / "runs" / RUN_NAME
    best_weights = RUN_DIR / "weights" / "best.pt"
    final_weights = MODELS_DIR / f"{RUN_NAME}_best.pt"
    shutil.copy2(best_weights, final_weights)
    print("Best weights ->", final_weights)
else:
    print("RUN_TRAINING is False; skipping training.")

## 10. Quick validation of the best checkpoint

In [ ]:
if RUN_TRAINING:
    from ultralytics import YOLO

    trained_model = YOLO(str(final_weights))
    val_metrics = trained_model.val(
        data=str(DATASET_YAML),
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=device,
        project=str(RESULTS_DIR / "runs"),
        name=f"{RUN_NAME}_val",
        exist_ok=True,
    )

    print(f"mAP@0.5      : {val_metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95 : {val_metrics.box.map:.4f}")
    print(f"Precision    : {val_metrics.box.mp:.4f}")
    print(f"Recall       : {val_metrics.box.mr:.4f}")
else:
    print("RUN_TRAINING is False; skipping validation.")

## 11. Loss and mAP curves

In [ ]:
if RUN_TRAINING:
    results_csv = RUN_DIR / "results.csv"
    if results_csv.exists():
        history = pd.read_csv(results_csv)
        history.columns = [column.strip() for column in history.columns]

        fig, axes = plt.subplots(1, 4, figsize=(20, 4))

        for ax, loss_name in zip(axes[:3], ("box_loss", "cls_loss", "dfl_loss")):
            ax.plot(history["epoch"], history[f"train/{loss_name}"], label="train")
            ax.plot(history["epoch"], history[f"val/{loss_name}"], label="val")
            ax.set_title(loss_name)
            ax.set_xlabel("epoch")
            ax.grid(alpha=0.3)
            ax.legend()

        axes[3].plot(history["epoch"], history["metrics/mAP50(B)"], label="mAP@0.5")
        axes[3].plot(
            history["epoch"],
            history["metrics/mAP50-95(B)"],
            label="mAP@0.5:0.95",
        )
        axes[3].set_title("Validation mAP")
        axes[3].set_xlabel("epoch")
        axes[3].grid(alpha=0.3)
        axes[3].legend()

        plt.tight_layout()
        plt.show()

        last = history.iloc[-1]
        print(
            f"Last epoch: mAP@0.5={last['metrics/mAP50(B)']:.4f} | "
            f"mAP@0.5:0.95={last['metrics/mAP50-95(B)']:.4f}"
        )
    else:
        print("No results.csv found; training was skipped.")
else:
    print("RUN_TRAINING is False; skipping curves.")

## 12. Next steps

- **Better accuracy:** set `QUICK_RUN = False`, `EPOCHS = 50`, and optionally
  switch `MODEL_NAME` to `yolov8s.pt` (~78–82% mAP@0.5 expected).
- **Evaluate:** open `02-evaluation.ipynb` for PR curves, per-class mAP, and
  the confusion matrix.
- **Inspect errors:** open `03-error-analysis.ipynb` for false positives and
  false negatives.
- **Run inference:** open `04-predict.ipynb` or use `app.py` locally.

**Label format reminder** — each label line is:

```
<class_id> <x_center> <y_center> <width> <height>
```

All coordinates are normalized to `[0, 1]` relative to the image size.